# Explore the FULFILL sufficiency model

Choose a background, year and measures. Follow real workbook assumptions through an ordered shock plan **without solving the IO system**. This notebook downloads only 1.75 MB of assumptions. The separate **Run_sufficiency.ipynb** notebook calculates the full model and a same-year control.

The historical paper files remain available through `fulfill reproduce`. Twenty-four BAU files differ from the declared recipe: this notebook uses the explicit composable recipe. See the verification guide.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from fulfill_model import Case, build_plan
from fulfill_model.data import ensure_data

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA = ROOT / 'Data'


## Choose assumptions

The six controls are D (diets), S (housing space), M (car size), P (shared machines), F (flights), B (cycling). A weight of 0.5 moves the affected instructions halfway from the same-year no-measure assumptions to the full-measure target.

In [2]:
BACKGROUND = 'NZE'
YEAR = 2050
MEASURES = ('D', 'F')
ADOPTION = {'F': 0.5}

case = Case(background=BACKGROUND, year=YEAR, measures=MEASURES,
            adoption=ADOPTION, name='diet-and-aviation-2050')
ensure_data(DATA, groups=('shocks',), progress=lambda _: None)
plan = build_plan(case, inputs=DATA / 'Files for shocks')
display(pd.Series(case.to_dict(), name='Selected case'))

name          diet-and-aviation-2050
background                       NZE
year                            2050
measures                      [D, F]
adoption                  {'F': 0.5}
Name: Selected case, dtype: object

## The ordered plan

Background evolution remains present. A sufficiency measure changes its associated blocks; paired effects remain coupled. The last step removes household purchases now represented by service activities.

In [3]:
display(plan.steps[['order','step','matrix','archetype','measure','adoption','rows']])

,order,step,matrix,archetype,measure,adoption,rows
0,1,cons_var,Y,scale_final_demand,background,1.0,86
1,2,car-eff,z,technical_requirement,M,0.0,3133
2,3,hs-eff,z,technical_requirement,background,1.0,2911
3,4,car-mix,z,supply_mix,background,1.0,135
4,5,hous,z,technical_requirement,background,1.0,144
5,6,hs-mix,z,supply_mix,background,1.0,216
6,7,ele-mix,z,electricity_mix,background,1.0,387
7,8,m2pc,Y,per_capita_demand,S,0.0,27
8,9,meals_dem,Y,total_service_demand,background,1.0,43
9,10,mvkm,Y,per_capita_demand,B,0.0,27


## A per-person assumption becomes total service demand

Car kilometres per person × population become a replacement demand for domestic car mobility. Source-cell references retain the workbook coordinates.

In [4]:
service = plan.rows[(plan.rows.step == 'mvkm') & (plan.rows['column region'] == 'IT')]
display(service[['input_file','input_cell','input_value','population','population_cell','value','type','row sector']])
assert service.iloc[0].value == service.iloc[0].input_value * service.iloc[0].population

,input_file,input_cell,input_value,population,population_cell,value,type,row sector
7096,mvkm_Full.xlsx,'vkm pc 0'!Q10,8350.541691,5.892921e+07,'Population_no_world'!I15,4.920908e+11,Update,Car mobility


## A technical requirement keeps its origin allocation

A fuel requirement per vehicle-kilometre is allocated across origins using the published baseline context. These weights come from net flows and can be signed; the diagnostic section exposes that limitation.

In [5]:
fuel = plan.rows[(plan.rows.step == 'car-eff') &
                 (plan.rows['column region'] == 'IT') &
                 (plan.rows['column sector'] == 'Diesel and gasoline car')]
display(fuel[['row region','input_cell','input_value','trade_share','value']].head(8))
assert (fuel.value == fuel.input_value * fuel.trade_share).all()

,row region,input_cell,input_value,trade_share,value
1704,AT,'car-eff 0 - clustering'!Q38,0.000039,1.776766e-03,6.962835e-08
1705,AU,'car-eff 0 - clustering'!Q38,0.000039,1.039179e-07,4.072363e-12
1706,BE,'car-eff 0 - clustering'!Q38,0.000039,1.995905e-03,7.821603e-08
1707,BG,'car-eff 0 - clustering'!Q38,0.000039,1.933179e-03,7.575790e-08
1708,BR,'car-eff 0 - clustering'!Q38,0.000039,3.107873e-06,1.217921e-10
1709,CA,'car-eff 0 - clustering'!Q38,0.000039,1.302462e-04,5.104122e-09
1710,CH,'car-eff 0 - clustering'!Q38,0.000039,2.394686e-05,9.384357e-10
1711,CN,'car-eff 0 - clustering'!Q38,0.000039,1.437981e-05,5.635198e-10


## Partial adoption changes instructions, then the model must be solved

Aviation acts on both household demand and intermediate use. The values below are fractional changes, not emission reductions. The two endpoints share the same background/year.

In [6]:
aviation = plan.rows[plan.rows.step.isin(['flying-less_Y','flying-less_z']) & (plan.rows['column region'] == 'IT')]
display(aviation[['step','reference_input_cell','input_cell','reference_value','target_value','adoption','value','type']])

,step,reference_input_cell,input_cell,reference_value,target_value,adoption,value,type
7797,flying-less_Y,'flying less - 0'!I15,'flying less - F'!I15,0.0,-0.736003,0.5,-0.368001,Percentage
7840,flying-less_z,'flying less - 0'!I15,'flying less - F'!I15,0.0,-0.736003,0.5,-0.368001,Percentage


## Diets change who supplies meals

Total meal demand and the supply mix are separate assumptions. These are real input shares, not calculated model impacts.

In [7]:
diets = plan.rows[(plan.rows.step == 'diet') & (plan.rows['column region'] == 'IT')]
display(diets[['row sector','value','input_cell']])
print('Meal supply-share total:', diets.value.sum())

,row sector,value,input_cell
7277,Omnivore 170g meal,0.081368,'diet D'!Q87
7278,Omnivore 75g meal,0.061772,'diet D'!Q88
7279,Flexitarian 30g meal,0.029784,'diet D'!Q89
7280,Vegetarian meal,0.010406,'diet D'!Q90
7281,Pescetarian meal,0.003861,'diet D'!Q91
7282,Vegan meal,0.000808,'diet D'!Q92
7283,Omnivore 100g opt meal,0.165992,'diet D'!Q93
7284,Omnivore 45g opt meal,0.253166,'diet D'!Q94
7285,Flexitarian 20g opt meal,0.204984,'diet D'!Q95
7286,Pescetarian opt meal,0.113539,'diet D'!Q96


Meal supply-share total: 1.0


## Preserve and diagnose the original inputs

Shares are not silently normalized. Two natural-gas markets lack an origin basis; 90 signed origin weights reflect the source net-flow data. The paper-compatible recipe preserves these conventions. These diagnostics matter when designing stricter, general-purpose shock archetypes.

In [8]:
totals = pd.DataFrame(plan.diagnostics['supply_mix_totals'])
display(totals.assign(deviation=(totals.total-1).abs()).sort_values('deviation',ascending=False).head(6))
display(pd.DataFrame(plan.diagnostics['no_baseline_market']))
print('Signed origin weights:', plan.diagnostics['negative_origin_weights'])

,step,region,commodity,total,deviation
4,car-mix,CY,Car mobility,1.029253,0.029253
18,car-mix,MT,Car mobility,1.029253,0.029253
32,hs-mix,CZ,Heating services,1.004232,0.004232
33,hs-mix,DK,Heating services,1.004232,0.004232
50,hs-mix,SK,Heating services,1.004232,0.004232
36,hs-mix,FR,Heating services,0.996469,0.003531


,commodity,consumer
0,Natural gas and services related to natural ga...,CY
1,Natural gas and services related to natural ga...,MT


Signed origin weights: 90


## Export the inspected plan

The destination is protected against accidental overwrites. Choose a new folder or explicitly use overwrite=True for a repeat. This step writes instructions only.

In [9]:
OUTPUT = ROOT / 'Outputs/plans/notebook-example'
plan.save(OUTPUT)
print('Saved ordered instructions, source ledger and diagnostics.')

Saved ordered instructions, source ledger and diagnostics.


## Run the full model next

Open **Run_sufficiency.ipynb** or use:

```bash
fulfill run --case config/diet-and-aviation.json --air-transport-extra \
  --output Outputs/runs/my-custom-case
```

This solves the selected case and its no-measure control. It can take several minutes and may download the 127.8 MB prepared database on first use. The output boundaries and aviation-account variant are explained in the README.